In [100]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


## import library

In [101]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

## path data

In [102]:
list_name_data=os.listdir(path)
list_name_data

['chest_xray']

In [103]:
path1=os.path.join(path,list_name_data[0])
path1

'/kaggle/input/chest-xray-pneumonia/chest_xray'

In [104]:
list_labels=os.listdir(path1)
list_labels

['chest_xray', '__MACOSX', 'val', 'test', 'train']

In [105]:
list_paths=[]
for i in list_labels:
    path2=os.path.join(path1,i)
    list_paths.append(path2)
list_paths

['/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/__MACOSX',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/val',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/test',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train']

In [106]:
path_val=list_paths[2]
path_test=list_paths[3]
path_train=list_paths[4]

In [107]:
path_test,path_train,path_val

('/kaggle/input/chest-xray-pneumonia/chest_xray/test',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/val')

## generatore

In [108]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2, # Use 20% of the data for validation
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
 )

In [109]:
train_data=datagen.flow_from_directory(
    path_train,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    subset='training' # Specify this is for training data
)

Found 4173 images belonging to 2 classes.


In [110]:
val_data=datagen.flow_from_directory(
    path_train,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    subset='validation' # Specify this is for validation data
)

Found 1043 images belonging to 2 classes.


In [111]:
test_data=datagen.flow_from_directory(
    path_test,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=False # Set shuffle to False for consistent evaluation
)

Found 624 images belonging to 2 classes.


In [112]:
class mymodel(Model):
  def __init__(self) -> None:
     super().__init__()

     self.conv1=Conv2D(32,(3,3),padding="same",activation="relu",input_shape=(150,150,1))
     self.maxpool1=MaxPooling2D((2,2))

     self.conv2=Conv2D(64,(3,3),padding="same",activation="relu")
     self.maxpool2=MaxPooling2D((2,2))

     self.conv3=Conv2D(128,(3,3),padding="same",activation="relu")
     self.maxpool3=MaxPooling2D((2,2))

     self.flatten=Flatten()
     self.dense1=Dense(128,activation="relu")
     self.dropout=Dropout(0.5)
     self.dense2=Dense(64,activation="relu")
     self.dense3=Dense(32,activation="relu")
     self.dense4=Dense(1,activation="sigmoid")

  def call(self,inputs):
    x=self.conv1(inputs)
    x=self.maxpool1(x)

    x=self.conv2(x)
    x=self.maxpool2(x)

    x=self.conv3(x)
    x=self.maxpool3(x)

    x=self.flatten(x)
    x=self.dense1(x)
    x=self.dropout(x)
    x=self.dense2(x)
    x=self.dense3(x)
    x=self.dense4(x)
    return x

In [113]:
model=mymodel()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [114]:
model.summary()

Model: "mymodel_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [115]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [116]:
earlystopping=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [117]:
history=model.fit(train_data,epochs=50,validation_data=val_data,callbacks=[earlystopping],batch_size=150)

Epoch 1/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 68s 490ms/step - accuracy: 0.7956 - loss: 0.4510 - val_accuracy: 0.8552 - val_loss: 0.3471
Epoch 2/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 52s 394ms/step - accuracy: 0.8929 - loss: 0.2531 - val_accuracy: 0.9108 - val_loss: 0.2163
Epoch 3/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 51s 388ms/step - accuracy: 0.9128 - loss: 0.2263 - val_accuracy: 0.8993 - val_loss: 0.2354
Epoch 4/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 48s 366ms/step - accuracy: 0.9149 - loss: 0.2086 - val_accuracy: 0.9022 - val_loss: 0.2452
Epoch 5/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 48s 367ms/step - accuracy: 0.9154 - loss: 0.2148 - val_accuracy: 0.8849 - val_loss: 0.2553
Epoch 6/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 48s 366ms/step - accuracy: 0.9336 - loss: 0.1688 - val_accuracy: 0.9252 - val_loss: 0.1755
Epoch 7/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 47s 362ms/step - accuracy: 0.9375 - loss: 0.1698 - val_accuracy: 0.9300 - val_loss: 0.1661
Epoch 8/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 47s 362ms/step - accuracy: 0.9322 - loss: 0

In [118]:
model.evaluate(test_data)

20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 213ms/step - accuracy: 0.8189 - loss: 0.6350


[0.6349677443504333, 0.8189102411270142]

In [119]:
from sklearn.metrics import classification_report

predictions = (model.predict(test_data) > 0.5).astype(int).flatten()
print(classification_report(test_data.classes, predictions))

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 215ms/step
              precision    recall  f1-score   support

           0       0.95      0.53      0.68       234
           1       0.78      0.98      0.87       390

    accuracy                           0.81       624
   macro avg       0.86      0.76      0.77       624
weighted avg       0.84      0.81      0.80       624



In [120]:

model.save("model.keras")